In [3]:
import pandas as pd

df_infra = pd.read_csv(
    "/content/track4_school_infrastructure.csv"
)

df_infra.head()

,inspection_id,date,school_id,has_electricity,has_drinking_water,has_functional_toilet,has_boundary_wall,has_playground,inspector_name,remarks
0,INSP02966,2025/04/02,sch_0476,H,True,True,Yes,NaN,Fariq Tripathi,Good condition
1,INSP00970,13.05.2025,SCH0337,True,True,Yes,True,H,Unnati Kulkarni,Good condition
2,INSP01386,2025/06/26,SCH0127,True,True,False,Working,Functional,Girik Natt,Toilets locked
3,INSP01234,02-Jun-2025,SCH0372,False,No,True,True,Broken,Rudra Kurian,Good condition
4,INSP02997,2025/10/29,sch0517,True,True,True,True,False,Mohammed Dara,Average


In [4]:
df_infra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   inspection_id          3150 non-null   object
 1   date                   3150 non-null   object
 2   school_id              3150 non-null   object
 3   has_electricity        2980 non-null   object
 4   has_drinking_water     3004 non-null   object
 5   has_functional_toilet  3010 non-null   object
 6   has_boundary_wall      2973 non-null   object
 7   has_playground         3004 non-null   object
 8   inspector_name         2818 non-null   object
 9   remarks                2637 non-null   object
dtypes: object(10)
memory usage: 246.2+ KB


In [5]:
print("Dataset shape:", df_infra.shape)

print("\nMissing values:")
print(df_infra.isna().sum())

print("\nDuplicate rows:", df_infra.duplicated().sum())

Dataset shape: (3150, 10)

Missing values:
inspection_id              0
date                       0
school_id                  0
has_electricity          170
has_drinking_water       146
has_functional_toilet    140
has_boundary_wall        177
has_playground           146
inspector_name           332
remarks                  513
dtype: int64

Duplicate rows: 150


In [6]:
facility_columns = [
    "has_electricity",
    "has_drinking_water",
    "has_functional_toilet",
    "has_boundary_wall",
    "has_playground"
]

for column in facility_columns:
    print(f"\n{'=' * 50}")
    print(column)
    print(df_infra[column].value_counts(dropna=False))


has_electricity
has_electricity
True             1098
False             382
NaN               170
Y                 154
Yes               144
1                 141
H                 118
Haan              117
haan              104
Hai                99
Working            93
Available          89
Functional         71
N                  58
Nahi hai           57
na                 45
Nahi               44
No                 44
0                  43
Kharab             22
Not Available      21
Under Repair       20
Broken             16
Name: count, dtype: int64

has_drinking_water
has_drinking_water
True             1270
False             237
1                 183
Y                 176
Yes               158
NaN               146
H                 138
haan              134
Haan              117
Hai               114
Available          87
Functional         83
Working            81
na                 42
0                  34
No                 30
Nahi               29
N                  28


### Standardize Facility Values

Convert different text and numeric representations into `True` or `False`.
Keep missing values as missing because they indicate unknown information.

In [7]:
facility_columns = [
    "has_electricity",
    "has_drinking_water",
    "has_functional_toilet",
    "has_boundary_wall",
    "has_playground"
]

true_values = {
    "true", "yes", "y", "1",
    "h", "haan", "hai",
    "working", "available", "functional"
}

false_values = {
    "false", "no", "n", "0",
    "na", "nahi", "nahi hai",
    "kharab", "not available",
    "under repair", "broken"
}

for column in facility_columns:
    cleaned_values = df_infra[column].astype("string").str.strip().str.lower()

    df_infra[column] = cleaned_values.map(
        lambda value: (
            True if value in true_values
            else False if value in false_values
            else pd.NA
        )
    ).astype("boolean")

### Check Facility Cleaning

Review the cleaned facility columns and confirm that only `True`, `False`, or missing values remain.

In [8]:
for column in facility_columns:
    print(f"\n{column}")
    print(df_infra[column].value_counts(dropna=False))


has_electricity
has_electricity
True     2228
False     752
<NA>      170
Name: count, dtype: Int64

has_drinking_water
has_drinking_water
True     2541
False     463
<NA>      146
Name: count, dtype: Int64

has_functional_toilet
has_functional_toilet
True     2397
False     613
<NA>      140
Name: count, dtype: Int64

has_boundary_wall
has_boundary_wall
True     1806
False    1167
<NA>      177
Name: count, dtype: Int64

has_playground
has_playground
False    1519
True     1485
<NA>      146
Name: count, dtype: Int64


### Validate Facility Values

Check for invalid values after standardization.

In [9]:
for column in facility_columns:
    invalid_values = df_infra.loc[
        ~df_infra[column].isin([True, False]) & df_infra[column].notna(),
        column
    ].unique()

    print(f"{column}: {invalid_values}")

has_electricity: <BooleanArray>
[]
Length: 0, dtype: boolean
has_drinking_water: <BooleanArray>
[]
Length: 0, dtype: boolean
has_functional_toilet: <BooleanArray>
[]
Length: 0, dtype: boolean
has_boundary_wall: <BooleanArray>
[]
Length: 0, dtype: boolean
has_playground: <BooleanArray>
[]
Length: 0, dtype: boolean


### Clean Inspection IDs

Standardize inspection IDs by removing extra spaces and converting them to uppercase.

In [10]:
df_infra["inspection_id"] = (
    df_infra["inspection_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_infra["inspection_id"].head()

,inspection_id
0,INSP02966
1,INSP00970
2,INSP01386
3,INSP01234
4,INSP02997


### Clean Dates

Convert all date values into a consistent datetime format.
Invalid dates will become missing values.

In [11]:
df_infra["date"] = pd.to_datetime(
    df_infra["date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

df_infra["date"].head()

,date
0,2025-04-02
1,2025-05-13
2,2025-06-26
3,2025-06-02
4,2025-10-29


### Validate Dates

Check whether any invalid or missing dates remain after conversion.

In [12]:
print("Missing dates:", df_infra["date"].isna().sum())

print("\nDate range:")
print("Start:", df_infra["date"].min())
print("End:", df_infra["date"].max())

Missing dates: 0

Date range:
Start: 2025-01-04 00:00:00
End: 2026-12-03 00:00:00


### Clean School IDs

Standardize school IDs by removing spaces and converting them to uppercase.

In [13]:
df_infra["school_id"] = (
    df_infra["school_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_infra["school_id"].head()

,school_id
0,SCH_0476
1,SCH0337
2,SCH0127
3,SCH0372
4,SCH0517


### Validate School IDs

Check the format and missing values of school IDs.

In [14]:
print("Missing school IDs:", df_infra["school_id"].isna().sum())

print("\nUnique school IDs:", df_infra["school_id"].nunique())

print("\nSample school IDs:")
print(df_infra["school_id"].head(10).tolist())

Missing school IDs: 0

Unique school IDs: 1559

Sample school IDs:
['SCH_0476', 'SCH0337', 'SCH0127', 'SCH0372', 'SCH0517', 'SCH_0334', 'S0229', '0072', 'SCH0516', 'SCH_0182']


### Standardize School ID Format

Convert all school IDs into the format `SCH####`.

In [15]:
df_infra["school_id"] = (
    df_infra["school_id"]
    .str.replace("_", "", regex=False)
    .str.replace("SCH", "", regex=False)
    .str.replace("S", "", regex=False)
    .str.strip()
)

df_infra["school_id"] = (
    "SCH" + df_infra["school_id"].str.zfill(4)
)

df_infra["school_id"].head(10)

,school_id
0,SCH0476
1,SCH0337
2,SCH0127
3,SCH0372
4,SCH0517
5,SCH0334
6,SCH0229
7,SCH0072
8,SCH0516
9,SCH0182


### Check School ID Duplicates

Check whether the standardized school IDs contain unexpected duplicates.

In [16]:
print("Unique school IDs:", df_infra["school_id"].nunique())

print("\nMost frequent school IDs:")
print(df_infra["school_id"].value_counts().head(10))

Unique school IDs: 914

Most frequent school IDs:
school_id
SCH0229    11
SCH0590    11
SCH0265    11
SCH0221    11
SCH0509    11
SCH0199    10
SCH0024    10
SCH0151    10
SCH0096    10
SCH0268    10
Name: count, dtype: Int64


### Clean Inspector Names

Remove extra spaces and standardize inspector names.

In [17]:
df_infra["inspector_name"] = (
    df_infra["inspector_name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df_infra["inspector_name"].head(10)

,inspector_name
0,Fariq Tripathi
1,Unnati Kulkarni
2,Girik Natt
3,Rudra Kurian
4,Mohammed Dara
5,<NA>
6,Meera Sachar
7,Shivani Konda
8,Yachana Mody
9,Eiravati Choudhry


### Check Inspector Names

Count missing names and review the most frequent inspector names.

In [19]:
print("Missing inspector names:", df_infra["inspector_name"].isna().sum())

print("\nUnique inspector names:", df_infra["inspector_name"].nunique())

print("\nMost frequent inspector names:")
print(df_infra["inspector_name"].value_counts().head(10))

Missing inspector names: 332

Unique inspector names: 2664

Most frequent inspector names:
inspector_name
Dhruv Jain            2
Nicholas Keer         2
Netra Sule            2
Chanakya Raval        2
Vrinda Comar          2
Amaira Padmanabhan    2
Warda Kade            2
Bakhshi Sani          2
Nimrat Hayer          2
Suhani Baral          2
Name: count, dtype: Int64


### Clean Remarks

Remove extra spaces and convert blank remarks into missing values.

In [20]:
df_infra["remarks"] = (
    df_infra["remarks"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .replace("", pd.NA)
)

df_infra["remarks"].head(10)

,remarks
0,Good condition
1,Good condition
2,Toilets locked
3,Good condition
4,Average
5,No remarks
6,Needs repair
7,Toilets locked
8,Good condition
9,Good condition


### Check Duplicate Inspections

Identify duplicate rows before deciding how to handle them.

In [21]:
print("Duplicate rows:", df_infra.duplicated().sum())

df_infra[df_infra.duplicated(keep=False)].head(10)

Duplicate rows: 150


,inspection_id,date,school_id,has_electricity,has_drinking_water,has_functional_toilet,has_boundary_wall,has_playground,inspector_name,remarks
12,INSP00613,2025-10-04,SCH0092,True,True,True,<NA>,False,Maya Chada,Good condition
18,INSP00145,2025-11-22,SCH0215,False,True,True,True,False,Yachana Mangat,Toilets locked
26,INSP00292,2025-11-25,SCH0371,True,True,True,False,True,Chasmum Ganguly,No remarks
38,INSP01815,2025-05-01,SCH0240,False,False,True,True,False,Lavanya Choudhry,Average
40,INSP01412,2025-08-05,SCH0596,False,True,True,False,False,Sai Bhasin,Average
43,INSP01227,2025-06-29,SCH0398,True,False,<NA>,True,True,Isaiah Mangat,<NA>
49,INSP02987,2025-06-24,SCH-0131,<NA>,True,True,False,True,Kevin Gade,Needs repair
50,INSP00103,2026-02-24,SCH0022,True,True,True,True,True,Netra Ram,Average
64,INSP00195,2025-12-07,SCH0346,True,True,True,True,False,Nimrat Hayer,Toilets locked
69,INSP00045,2025-04-11,SCH-0003,True,True,False,True,True,Nicholas Keer,No remarks


### Remove Duplicate Inspection Records

Remove exact duplicate records while keeping the first occurrence.

In [22]:
df_infra = df_infra.drop_duplicates().reset_index(drop=True)

print("Remaining rows:", len(df_infra))
print("Duplicate rows:", df_infra.duplicated().sum())

Remaining rows: 3000
Duplicate rows: 0


### Final Dataset Validation

Review the cleaned data types, missing values, and duplicate records.

In [23]:
print(df_infra.info())

print("\nMissing values:")
print(df_infra.isna().sum())

print("\nDuplicate rows:", df_infra.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   inspection_id          3000 non-null   string        
 1   date                   3000 non-null   datetime64[ns]
 2   school_id              3000 non-null   string        
 3   has_electricity        2836 non-null   boolean       
 4   has_drinking_water     2861 non-null   boolean       
 5   has_functional_toilet  2867 non-null   boolean       
 6   has_boundary_wall      2833 non-null   boolean       
 7   has_playground         2860 non-null   boolean       
 8   inspector_name         2677 non-null   string        
 9   remarks                2512 non-null   string        
dtypes: boolean(5), datetime64[ns](1), string(4)
memory usage: 146.6 KB
None

Missing values:
inspection_id              0
date                       0
school_id                  0
has_

### Fix School ID Formatting

Remove hyphens and underscores from school IDs and standardize them as `SCH####`.

In [24]:
df_infra["school_id"] = (
    df_infra["school_id"]
    .astype("string")
    .str.upper()
    .str.replace(r"[-_]", "", regex=True)
    .str.replace("SCH", "", regex=False)
    .str.replace("S", "", regex=False)
    .str.strip()
)

df_infra["school_id"] = (
    "SCH" + df_infra["school_id"].str.zfill(4)
)

df_infra["school_id"].head(10)

,school_id
0,SCH0476
1,SCH0337
2,SCH0127
3,SCH0372
4,SCH0517
5,SCH0334
6,SCH0229
7,SCH0072
8,SCH0516
9,SCH0182


### Validate School IDs

Check whether any school IDs still contain hyphens, underscores, or invalid formats.

In [25]:
invalid_school_ids = df_infra.loc[
    ~df_infra["school_id"].str.match(r"^SCH\d{4}$", na=False),
    "school_id"
]

print("Invalid school IDs:", len(invalid_school_ids))
print(invalid_school_ids.unique())

Invalid school IDs: 0
<StringArray>
[]
Length: 0, dtype: string


### Save Cleaned Infrastructure Dataset

Save the cleaned infrastructure data as a CSV file.

In [27]:
import os

os.makedirs("../data/cleaned", exist_ok=True)

df_infra.to_csv(
    "../data/cleaned/infrastructure_cleaned.csv",
    index=False
)

print("Infrastructure dataset saved successfully.")

Infrastructure dataset saved successfully.
